# Baseline Risk Definition

## Purpose
This notebook establishes **baseline risk definitions** for identifying schools with low graduation outcomes.
Before introducing machine learning models, simple rule-based baselines are evaluated to:

- Understand class imbalance
- Establish naive performance benchmarks
- Test whether intuitive hueristics can surface risk signals
- Define an intepretable starting point for later models

All analysis is performed on the curated dbt mart: mart_grad_rate_risk




## 1. Connect to Duckdb

In [16]:
import duckdb
import pandas as pd

DB_PATH = "../accountability_project/dev.duckdb"
con = duckdb.connect(DB_PATH)

con.execute("""
select schema_name
from information_schema.schemata
""").fetchdf()

,schema_name
0,main
1,information_schema
2,main
3,pg_catalog
4,main


## 2. Validate Mart Schema
Before defining baselines, confirm the structure of the modeling table.

In [17]:
con.execute("describe main.mart_grad_rate_risk").fetch_df()

,column_name,column_type,null,key,default,extra
0,district_number,VARCHAR,YES,None,None,None
1,district_name,VARCHAR,YES,None,None,None
2,school_number,VARCHAR,YES,None,None,None
3,school_name,VARCHAR,YES,None,None,None
4,school_type,INTEGER,YES,None,None,None
5,title_i,VARCHAR,YES,None,None,None
6,pct_econ_disadvantaged,DOUBLE,YES,None,None,None
7,grade_2025,VARCHAR,YES,None,None,None
8,grade_2024,VARCHAR,YES,None,None,None
9,grade_2025_score,INTEGER,YES,None,None,None


## 3. Row Counts and Duplicate Schools
The mart currently contains **multiple rows per school**.
This is expected based on upstream joins and is explicitly examined here.

In [18]:
query = """
select
    count(*) as n_rows,
    count(distinct school_number) as n_district_schools,
    count(*) - count(distinct school_number) as n_duplicate_school_rows
from main.mart_grad_rate_risk;
"""
con.execute(query).fetch_df()

,n_rows,n_district_schools,n_duplicate_school_rows
0,3465,1225,2240


## 4. Target Variable Distribution

### Raw Counts

In [19]:
query = """
select
    is_grad_rate_risk,
    count(*) as n
from main.mart_grad_rate_risk
group by 1
order by 1;
"""
con.execute(query).fetch_df()

,is_grad_rate_risk,n
0,0,3445
1,1,20


### Percent Distribution

In [5]:
query = """
with base as (
    select
        is_grad_rate_risk,
        count(*) as n
    from main.mart_grad_rate_risk
    group by 1
)
select
    is_grad_rate_risk,
    n,
    round(100.0 * n / sum(n) over (), 2) as pct
from base
order by is_grad_rate_risk;
"""
con.execute(query).fetch_df()

,is_grad_rate_risk,n,pct
0,0,3445,99.42
1,1,20,0.58


### Observation:
The dataset is **extremely imbalanced**, with fewer thant 1% of rows labeled as at-risk.

## 5. Naive Majority-Class Baseline
As a reference point, consider a classifier that **predicts all schools as not at risk**.

In [20]:
query = """
select
    sum(case when is_grad_rate_risk = 0 then 1 else 0 end) as majority_class_n,
    count(*) as total_n,
    round(
        100.0 * sum(case when is_grad_rate_risk = 0 then 1 else 0 end) / count(*),
        2
    ) as majority_class_accuracy_pct
from main.mart_grad_rate_risk;
"""
con.execute(query).fetch_df()

,majority_class_n,total_n,majority_class_accuracy_pct
0,3445.0,3465,99.42


This extablishes a **high-accuracy but zero-recall baseline**, which is common in rare-event problems.

## 6. Confusion Matrix for Naive Baseline

In [21]:
query = """
select
    0 as tp,
    0 as fp,
    sum(case when is_grad_rate_risk = 1 then 1 else 0 end) as fn,
    sum(case when is_grad_rate_risk = 0 then 1 else 0 end) as tn
from main.mart_grad_rate_risk;
"""
con.execute(query).fetch_df()

,tp,fp,fn,tn
0,0,0,20.0,3445.0


## 7. Precision/Recall for Naive Baseline

In [22]:
query = """
with cm as (
    select
        0 as tp,
        0 as fp,
        sum(case when is_grad_rate_risk = 1 then 1 else 0 end) as fn,
        sum(case when is_grad_rate_risk = 0 then 1 else 0 end) as tn
    from main.mart_grad_rate_risk
)
select
    tp, fp, fn, tn,
    null as precision,
    round(1.0 * tp / nullif(tp + fn, 0), 4) as recall,
    round(1.0 * (tp + tn) / (tp + fp + fn + tn), 4) as accuracy
from cm;
"""
con.execute(query).fetch_df()

,tp,fp,fn,tn,precision,recall,accuracy
0,0,0,20.0,3445.0,<NA>,0.0,0.9942


## 8. Rule-Based Baseline Definitions
Three simple, interpretable rules are evaluated:
- **Rule A**: Sharp grade decline and high economic disadvantage
- **Rule B**: Extremely high economic disadvantage
- **Rule C**: currernt letter grade of D or F

In [23]:
query = """
select
    school_number,
    school_name,
    is_grad_rate_risk,
    pct_econ_disadvantaged,
    grade_2025_score,
    grade_delta_2025_vs_2024,

    case
        when grade_delta_2025_vs_2024 <= -1
        and pct_econ_disadvantaged >= 75
        then 1 else 0
    end as pred_rule_a,

    case
        when pct_econ_disadvantaged >= 90
        then 1 else 0
    end as pred_rule_b,

    case
        when grade_2025_score <= 1
        then 1 else 0
    end as pred_rule_c

from main.mart_grad_rate_risk
limit 25;
"""
con.execute(query).fetch_df()

,school_number,school_name,is_grad_rate_risk,pct_econ_disadvantaged,grade_2025_score,grade_delta_2025_vs_2024,pred_rule_a,pred_rule_b,pred_rule_c
0,0031,CAROLYN BEATRICE PARKER ELEMENTARY,0,66.1,3,0,0,0,0
1,0041,STEPHEN FOSTER ELEMENTARY SCHOOL,0,100.0,1,-1,1,1,1
2,0071,LAKE FOREST ELEMENTARY SCHOOL,0,100.0,2,0,0,1,0
3,0091,LITTLEWOOD ELEMENTARY SCHOOL,0,75.0,4,1,0,0,0
4,0101,W. A. METCALFE ELEMENTARY SCHOOL,0,100.0,2,0,0,1,0
5,0111,JOSEPH WILLIAMS ELEMENTARY SCHOOL,0,100.0,1,-2,1,1,1
6,0112,ABRAHAM LINCOLN MIDDLE SCHOOL,0,89.8,3,-1,1,0,0
7,0121,HOWARD W. BISHOP MIDDLE SCHOOL,0,83.9,3,1,0,0,0
8,0141,WESTWOOD MIDDLE SCHOOL,0,81.7,2,0,0,0,0
9,0151,GAINESVILLE HIGH SCHOOL,0,47.6,4,1,0,0,0


## 9. Rule-Based Performance (Row-Level)

In [24]:
query = """
with preds as (
    select
        is_grad_rate_risk as y,

        case
            when grade_delta_2025_vs_2024 <= -1
            and pct_econ_disadvantaged >= 75
            then 1 else 0
        end as pred_rule_a,

        case
            when pct_econ_disadvantaged >= 90
            then 1 else 0
        end as pred_rule_b,

        case
            when grade_2025_score <= 1
            then 1 else 0
        end as pred_rule_c

    from main.mart_grad_rate_risk
),
eval as (
    select 'rule_a' as rule, pred_rule_a as pred, y from preds
    union all
    select 'rule_b', pred_rule_b, y from preds
    union all
    select 'rule_c', pred_rule_c, y from preds
),
cm as (
    select
        rule,
        sum(case when pred = 1 and y = 1 then 1 else 0 end) as tp,
        sum(case when pred = 1 and y = 0 then 1 else 0 end) as fp,
        sum(case when pred = 0 and y = 1 then 1 else 0 end) as fn,
        sum(case when pred = 0 and y = 0 then 1 else 0 end) as tn,
        count(*) as n_total
    from eval
    group by rule
)
select
    rule,
    tp, fp, fn, tn,
    round(1.0 * tp / nullif(tp + fp, 0), 4) as precision,
    round(1.0 * tp / nullif(tp + fn, 0), 4) as recall,
    round(1.0 * (tp + tn) / (tp + fp + fn + tn), 4) as accuracy
from cm
order by rule;
"""
con.execute(query).fetch_df()

,rule,tp,fp,fn,tn,precision,recall,accuracy
0,rule_a,0.0,238.0,20.0,3207.0,0.0000,0.00,0.9255
1,rule_b,11.0,1411.0,9.0,2034.0,0.0077,0.55,0.5902
2,rule_c,4.0,66.0,16.0,3379.0,0.0571,0.20,0.9763


## 10. One-Row-Per-School Aggregation Strategy
For modeling and evaluation, schools, not rows, should be the unit of prediction.

A conservative aggregation is applied:
- **Risk label**: max(is_grad_rate_risk)
- **Continuous features**: max() to avoid making risk
- **Identifiers**: any_value()

In [25]:
query = """
with school_level as (
    select
        school_number,
        any_value(school_name) as school_name,
        max(is_grad_rate_risk) as is_grad_rate_risk,
        max(pct_econ_disadvantaged) as pct_econ_disadvantaged,
        max(grade_2025_score) as grade_2025_score,
        max(grade_delta_2025_vs_2024) as grade_delta_2025_vs_2024
    from main.mart_grad_rate_risk
    group by school_number
)
select
    count(*) as n_schools,
    sum(is_grad_rate_risk) as n_at_risk
from school_level;
"""
con.execute(query).fetch_df()

,n_schools,n_at_risk
0,1225,17.0


## 11. Rule Performance at School Level

In [26]:
query = """
with school_level as (
    select
        school_number,
        max(is_grad_rate_risk) as y,
        max(pct_econ_disadvantaged) as pct_econ_disadvantaged,
        max(grade_2025_score) as grade_2025_score,
        max(grade_delta_2025_vs_2024) as grade_delta_2025_vs_2024
    from main.mart_grad_rate_risk
    group by school_number
),
preds as (
    select
        y,
        case
            when grade_delta_2025_vs_2024 <= -1
            and pct_econ_disadvantaged >= 75
            then 1 else 0
        end as pred_rule_a,

        case
            when pct_econ_disadvantaged >= 90
            then 1 else 0
        end as pred_rule_b,

        case
            when grade_2025_score <= 1
            then 1 else 0
        end as pred_rule_c
    from school_level
),
eval as (
    select 'rule_a' as rule, pred_rule_a as pred, y from preds
    union all
    select 'rule_b', pred_rule_b, y from preds
    union all
    select 'rule_c', pred_rule_c, y from preds
),
cm as (
    select
        rule,
        sum(case when pred = 1 and y = 1 then 1 else 0 end) as tp,
        sum(case when pred = 1 and y = 0 then 1 else 0 end) as fp,
        sum(case when pred = 0 and y = 1 then 1 else 0 end) as fn,
        sum(case when pred = 0 and y = 0 then 1 else 0 end) as tn
    from eval
    group by rule
)
select
    rule,
    tp, fp, fn, tn,
    round(1.0 * tp / nullif(tp + fp, 0), 4) as precision,
    round(1.0 * tp / nullif(tp + fn, 0), 4) as recall,
    round(1.0 * (tp + tn) / (tp + fp + fn + tn), 4) as accuracy
from cm
order by rule;
"""
con.execute(query).fetch_df()

,rule,tp,fp,fn,tn,precision,recall,accuracy
0,rule_a,0.0,34.0,17.0,1174.0,0.0000,0.0000,0.9584
1,rule_b,12.0,657.0,5.0,551.0,0.0179,0.7059,0.4596
2,rule_c,1.0,12.0,16.0,1196.0,0.0769,0.0588,0.9771


## 12. One-Row-Per-School Aggregation
Before evaluating rule-based risk signals, the dataset is collapsed to a single row per school.
This ensures that each school contributes exactly one outcome tp downstream evaluation, preventing duplicate rows from inflating metric or distorting class balance.

In [28]:
query = """
with school_level as (
    select
        school_number,
        max(is_grad_rate_risk) as y,
        max(pct_econ_disadvantaged) as pct_econ_disadvantaged,
        max(grade_2025_score) as grade_2025_score,
        max(grade_delta_2025_vs_2024) as grade_delta_2025_vs_2024
    from main.mart_grad_rate_risk
    group by school_number
),
base as (
    select
        sum(y) as n_pos,
        count(*) as n_total,
        1.0 * sum(y) / count(*) as base_rate
    from school_level
),
preds as (
    select
        *,
        case when grade_delta_2025_vs_2024 <= -1 and pct_econ_disadvantaged >= 75 then 1 else 0 end as pred_rule_a,
        case when pct_econ_disadvantaged >= 90 then 1 else 0 end as pred_rule_b,
        case when grade_2025_score <= 1 then 1 else 0 end as pred_rule_c
    from school_level
),
long as (
    select 'rule_a' as rule, pred_rule_a as pred, y from preds
    union all select 'rule_b', pred_rule_b, y from preds
    union all select 'rule_c', pred_rule_c, y from preds
),
agg as (
    select
        rule,
        sum(pred) as n_flagged,
        sum(case when pred = 1 and y = 1 then 1 else 0 end) as tp,
        sum(case when pred = 1 and y = 0 then 1 else 0 end) as fp,
        sum(case when pred = 0 and y = 1 then 1 else 0 end) as fn,
        sum(case when pred = 0 and y = 0 then 1 else 0 end) as tn
    from long
    group by rule
)
select
    a.rule,
    a.n_flagged,
    a.tp, a.fp, a.fn, a.tn,
    round(1.0 * a.tp / nullif(a.tp + a.fp, 0), 4) as precision,
    round(1.0 * a.tp / nullif(a.tp + a.fn, 0), 4) as recall,
    round(1.0 * a.n_flagged / (a.tp + a.fp + a.fn + a.tn), 4) as flagged_rate,
    round((1.0 * a.tp / nullif(a.n_flagged, 0)) / b.base_rate, 2) as lift_vs_base
from agg a
cross join base b
order by rule;
"""

con.execute(query).fetch_df()

,rule,n_flagged,tp,fp,fn,tn,precision,recall,flagged_rate,lift_vs_base
0,rule_a,34.0,0.0,34.0,17.0,1174.0,0.0000,0.0000,0.0278,0.00
1,rule_b,669.0,12.0,657.0,5.0,551.0,0.0179,0.7059,0.5461,1.29
2,rule_c,13.0,1.0,12.0,16.0,1196.0,0.0769,0.0588,0.0106,5.54


In [14]:
con.close()

Multiple rule-based baselines were evaluated to establish recall-precision tradeoffs under severe class imbalance.